# Texture, ODF Evaluation, And Pole-Figure Inversion

This notebook demonstrates the current PyTex texture model:

- pole-figure synthesis from orientation sets
- inverse-pole-figure synthesis
- kernel-weighted ODF evaluation
- discrete pole-figure inversion through an explicit orientation dictionary
- contour pole figures and Bunge-section ODF plotting through the runtime API

## Discrete Forward Model

For measurement directions $\mathbf{s}_m$ and dictionary orientations $g_j$, the
current inversion builds a forward matrix

$$
A_{mj} = \frac{1}{|\mathcal{H}|}\sum_{h\in\mathcal{H}} K\bigl(\angle(\mathbf{s}_m, g_j h)\bigr),
$$

then solves a regularized nonnegative least-squares problem over the dictionary
weights.


In [ ]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalCellOverlay,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalMap,
    CrystalPlane,
    CrystalPlaneOverlay,
    DirectionAnnotationStyle,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    get_phase_fixture,
    list_phase_fixtures,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    read_validation_manifest,
    read_workflow_result_manifest,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    PlaneAnnotationStyle,
    generate_saed_pattern,
    generate_xrd_pattern,
    normalize_ebsd,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_ipf_map,
    plot_orientations,
    plot_kam_map,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_crystal_frame():
    return ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )


def make_context():
    crystal = make_crystal_frame()
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    phase = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


def describe_phase_fixture(fixture_id):
    record = get_phase_fixture(fixture_id)
    return {
        "fixture_id": record.fixture_id,
        "display_name": record.display_name,
        "artifact_path": str(record.artifact_path),
        "metadata_path": str(record.metadata_path),
        "intended_uses": tuple(record.metadata["intended_uses"]),
    }


def load_zr_hcp_phase():
    return get_phase_fixture("zr_hcp").load_phase(crystal_frame=make_crystal_frame())


def load_diamond_phase():
    return get_phase_fixture("diamond").load_phase(crystal_frame=make_crystal_frame())


def publication_crystal_style():
    return {
        "crystal": {
            "atom_radius_scale": 0.5,
            "atom_edgewidth": 0.0,
            "atom_surface_resolution": 34,
            "bond_surface_resolution": 28,
            "bond_alpha": 0.72,
            "bond_color": "#7c8ea3",
            "atom_specular_strength": 0.42,
            "light_specular": 0.4,
        }
    }


In [ ]:
crystal, specimen, *_ , phase = make_context()
symmetry = phase.symmetry

support = OrientationSet.from_euler_angles(
    np.array(
        [
            [0.0, 0.0, 0.0],
            [90.0, 0.0, 0.0],
            [35.0, 25.0, 10.0],
        ]
    ),
    crystal_frame=crystal,
    specimen_frame=specimen,
    symmetry=symmetry,
    phase=phase,
)

odf = ODF.from_orientations(
    support,
    weights=[4.0, 2.0, 1.0],
    kernel=KernelSpec(name="von_mises_fisher", halfwidth_deg=10.0),
)

query = Orientation(
    rotation=Rotation.identity(),
    crystal_frame=crystal,
    specimen_frame=specimen,
    symmetry=symmetry,
    phase=phase,
)
print("ODF(query) =", odf.evaluate(query))
print("Volume fraction within 15 deg =", odf.volume_fraction(query, max_angle_deg=15.0))


In [ ]:
poles = (
    CrystalPlane(miller=MillerIndex([1, 0, 0], phase=phase), phase=phase),
    CrystalPlane(miller=MillerIndex([1, 1, 1], phase=phase), phase=phase),
)

pole_figures = odf.reconstruct_pole_figures(
    poles,
    include_symmetry_family=False,
    antipodal=False,
)

inversion = ODF.invert_pole_figures(
    pole_figures,
    orientation_dictionary=support,
    kernel=odf.kernel,
    regularization=1e-8,
    include_symmetry_family=False,
)

print("Converged:", inversion.converged)
print("Estimated weights:", inversion.odf.normalized_weights)
print("Residual norm:", inversion.residual_norm)


In [ ]:
ipf = InversePoleFigure.from_orientations(
    support,
    [0.0, 0.0, 1.0],
    reduce_by_symmetry=True,
    antipodal=True,
)
print(ipf.crystal_directions)


In [ ]:
contour_pf = plot_pole_figure(
    pole_figures[0],
    kind="contour",
    bins=81,
    sigma_bins=1.5,
    levels=12,
    title="Pole Figure Contours",
)
odf_sections = plot_odf(
    inversion.odf,
    kind="sections",
    section_phi2_deg=(0.0, 15.0, 30.0, 45.0, 60.0, 75.0),
    section_phi1_steps=121,
    section_big_phi_steps=61,
    levels=10,
    title="Estimated ODF Bunge Sections",
)
contour_pf


In [ ]:
odf_sections


The contour pole figure is built from a smoothed projected density grid over the
reconstructed pole data. The ODF section plot is a kernel-smoothed Bunge-section
inspection view over the discrete support rather than a harmonic ODF expansion.
